# Distilling Wikontic Full Pipeline into SmolLM2-1.7B

**Goal:** Train a small local model (SmolLM2-1.7B-Instruct) to replace the entire Wikontic pipeline -- taking raw text and producing ontology-aligned triplets in a single forward pass.

**Data:**
- datasets/hotpotqa200.json -- source texts (200 samples, ~10 paragraphs each)
- datasets/kg_dump_hotpot_gpt4_1_onto_triplets.json -- ground-truth ontology-aligned triplets produced by GPT-4.1 via the full Wikontic pipeline

**Approach:** QLoRA fine-tuning with trl.SFTTrainer. Each training example is a chat-formatted (text -> triplets) pair derived from the same prompt that Wikontic uses for extraction.

In [1]:
# Install missing dependencies (trl, peft, bitsandbytes, datasets)
! pip install -q trl peft bitsandbytes datasets



[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## 1. Load and Prepare Data

In [2]:
import json
from pathlib import Path
import os
import sys
os.environ['PYTHONUTF8'] = '1'

HOTPOT_PATH = Path("../datasets/hotpotqa200.json")
DUMP_PATH   = Path("../datasets/kg_dump_hotpot_gpt4_1_onto_triplets.json")
SYSTEM_PROMPT_PATH = Path("../src/wikontic/utils/prompts/triplet_extraction/propmt_1_types_qualifiers.txt")

OUTPUT_DIR = Path("./data")
TRAIN_OUT = OUTPUT_DIR / "train.jsonl"
VAL_OUT   = OUTPUT_DIR / "val.jsonl"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Load hotpotqa -- indexed by sample_id
with open(HOTPOT_PATH) as f:
    hotpot = {s["_id"]: s for s in json.load(f)}

# Load kg_dump -- same structure: {sample_id: {source_id: {triplets, ...}}}
with open(DUMP_PATH) as f:
    dump = json.load(f)

print(f"HotPotQA samples: {len(hotpot)}")
print(f"KG dump samples:   {len(dump)}")
print(f"Overlapping IDs:   {len(set(hotpot) & set(dump))}")

# Load system prompt
SYSTEM_PROMPT = open(SYSTEM_PROMPT_PATH, encoding="utf-8").read()
print(f"\nSystem prompt length: {len(SYSTEM_PROMPT)} chars")

HotPotQA samples: 200
KG dump samples:   200
Overlapping IDs:   200

System prompt length: 4034 chars


## 2. Inspect Data Structure

In [3]:
# Inspect a single example
sample_id = list(dump.keys())[0]
sample = hotpot[sample_id]
entry  = dump[sample_id]["0"]   # first paragraph

print("=== HotPotQA sample ===")
print("Question:", sample["question"])
print("Answer:",   sample["answer"])
print("Context count:", len(sample["context"]))
title, chunks = sample["context"][0]
print("First paragraph title:", title)
print("First paragraph text:", " ".join(chunks)[:200], "...")

print("\n=== KG dump for this paragraph ===")
print("Triplets count:", len(entry["triplets"]))
for t in entry["triplets"][:3]:
    print(" -", t)

=== HotPotQA sample ===
Question: Which of these universities, Northwestern University or Johns Hopkins University, have a campus outside of the United States territories?
Answer: with other campuses located in Chicago and Doha, Qatar
Context count: 10
First paragraph title: Northwestern University
First paragraph text: Northwestern University (NU) is a private research university based in Evanston, Illinois, with other campuses located in Chicago and Doha, Qatar, and academic programs and facilities in Washington, D ...

=== KG dump for this paragraph ===
Triplets count: 6
 - {'object': 'private research university', 'object_type': 'educational institution', 'relation': 'instance of', 'subject': 'Northwestern University', 'subject_type': 'university', 'qualifiers': []}
 - {'object': 'Evanston, Illinois', 'object_type': 'city', 'relation': 'headquarters location', 'subject': 'Northwestern University', 'subject_type': 'university', 'qualifiers': []}
 - {'object': 'Chicago', 'object_type

## 3. Build Training Examples

In [4]:
def build_example(text: str, triplets: list, system_prompt: str) -> dict:
    """Format one text-triplets pair as a flat SFT string.
    
    The 'text' field concatenates system + user + assistant messages.
    This is required for dataset_text_field in SFTTrainer."""
    assistant_content = json.dumps({"triplets": triplets}, ensure_ascii=False)
    full_text = (
        f"{system_prompt}\n\n"
        f'Text: "{text}"\n\n'
        f"{assistant_content}"
    )
    return {"text": full_text}


examples = []
skipped_empty = 0

for sample_id, source_dict in dump.items():
    if sample_id not in hotpot:
        continue
    sample = hotpot[sample_id]
    context = sample["context"]   # list of [title, [text_segments]]

    for sid_str, entry in source_dict.items():
        sid = int(sid_str)
        if sid >= len(context):
            continue

        title, text_segments = context[sid]
        text = " ".join(text_segments).strip()
        triplets = entry.get("triplets", [])

        if not text or not triplets:
            skipped_empty += 1
            continue

        examples.append(build_example(text, triplets, SYSTEM_PROMPT))

print(f"Total examples: {len(examples)}")
print(f"Skipped (empty text or triplets): {skipped_empty}")


Total examples: 1999
Skipped (empty text or triplets): 0


In [5]:
# Basic statistics
def count_triplets(example_text):
    """Extract JSON object from text and return triplet count."""
    try:
        start = example_text.rfind('{"triplets":')
        if start < 0:
            return 0
        json_start = example_text.find('[', start)
        if json_start < 0:
            return 0
        depth = 1
        pos = json_start + 1
        while pos < len(example_text) and depth > 0:
            c = example_text[pos]
            if c == '[':
                depth += 1
            elif c == ']':
                depth -= 1
            pos += 1
        if pos < len(example_text) and example_text[pos] == '}':
            candidate = example_text[start:pos + 1]
            return len(json.loads(candidate)['triplets'])
        return 0
    except Exception:
        return 0

triplet_counts = [count_triplets(e['text']) for e in examples]
non_zero = sum(1 for c in triplet_counts if c > 0)
print(f"Min / Max / Avg triplets per example: {min(triplet_counts)} / {max(triplet_counts)} / {sum(triplet_counts)/len(triplet_counts):.1f}")
print(f"Non-zero: {non_zero} / {len(triplet_counts)}")

# Show one formatted example
ex = examples[10]
print()
print("=== Example #10 (first 200 chars of text field) ===")
print(ex['text'][:200])


Min / Max / Avg triplets per example: 1 / 48 / 11.2
Non-zero: 1999 / 1999

=== Example #10 (first 200 chars of text field) ===
You are an algorithm designed to extract structured knowledge from texts to build a Wikidata-like knowledge graph. A knowledge graph consists of **triplets** in the format (subject, relation, object),


## 4. Train / Val Split and Save

In [6]:
import random

random.seed(42)
random.shuffle(examples)

val_size = max(1, int(len(examples) * 0.1))
val_examples   = examples[:val_size]
train_examples = examples[val_size:]

def write_jsonl(path: Path, items: list):
    with open(path, "w", encoding="utf-8") as f:
        for item in items:
            f.write(json.dumps(item, ensure_ascii=False) + "\n")

write_jsonl(TRAIN_OUT, train_examples)
write_jsonl(VAL_OUT,   val_examples)

print(f"Train: {len(train_examples)} | Val: {len(val_examples)}")
print(f"Saved to {TRAIN_OUT} and {VAL_OUT}")

Train: 1800 | Val: 199
Saved to data\train.jsonl and data\val.jsonl


## 5. Load Model -- SmolLM2-1.7B-Instruct with QLoRA

In [7]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from transformers import BitsAndBytesConfig

BASE_MODEL = "HuggingFaceTB/SmolLM2-1.7B-Instruct"

# 4-bit NF4 quantization
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
)

print("Loading base model...")
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto",
)
model = prepare_model_for_kbit_training(model)

# LoRA adapter
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q_proj", "v_proj"],
    task_type="CAUSAL_LM",
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

# Tokenizer
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
tokenizer.pad_token = tokenizer.eos_token

D:\PycharmProjects\Wikontic\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading base model...


Loading weights: 100%|██████████| 218/218 [00:00<00:00, 230.09it/s]


trainable params: 3,145,728 || all params: 1,714,522,112 || trainable%: 0.1835


## 6. Configure and Run SFT Training

In [ ]:
# Fix encoding issue on Windows before importing trl
from pathlib import Path
import sys
sys.path.insert(0, str(Path(".").resolve()))
import trl_utf8_fix  # noqa: E402

from trl import SFTTrainer
from transformers import TrainingArguments
from datasets import load_dataset

CHECKPOINT_DIR = "./checkpoints/wikontic-full-pipeline"

# Load datasets
from datasets import Dataset, DatasetDict

def load_jsonl(path):
    with open(path, "r", encoding="utf-8") as f:
        return [json.loads(line) for line in f]

ds = DatasetDict({
    "train": Dataset.from_list(load_jsonl(TRAIN_OUT)),
    "val":   Dataset.from_list(load_jsonl(VAL_OUT)),
})
print(f"Train: {len(ds['train'])} | Val: {len(ds['val'])}")


# Training arguments
training_args = TrainingArguments(
    output_dir=CHECKPOINT_DIR,
    num_train_epochs=3,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,       # effective batch = 16
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_steps=100,
    max_grad_norm=0.3,
    bf16=True,
    logging_steps=10,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=1,
    report_to="none",
)

# SFT Trainer - model is already a PeftModel
trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=ds["train"],
    eval_dataset=ds["val"],
    processing_class=tokenizer,  # new API: processing_class instead of tokenizer
    formatting_func=lambda x: x["text"],
)

print("Starting training...")
trainer.train()




Train: 1800 | Val: 199


Tokenizing eval dataset: 100%|██████████| 199/199 [00:00<00:00, 593.26 examples/s]


Starting training...


D:\PycharmProjects\Wikontic\.venv\Lib\site-packages\torch\_dynamo\eval_frame.py:632: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


## 7. Save Model

In [ ]:
trainer.save_model(CHECKPOINT_DIR)
tokenizer.save_pretrained(CHECKPOINT_DIR)
print(f"Model saved to {CHECKPOINT_DIR}")

## 8. Quick Validation -- Inference on a Few Examples

In [ ]:
from peft import PeftModel
import re

# Reload base model and apply the LoRA adapter
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto",
)
model_to_eval = PeftModel.from_pretrained(base_model, CHECKPOINT_DIR)
model_to_eval.eval()

def extract_json(text):
    """Try to parse JSON from model output."""
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        match = re.search(r'\{.+\}', text, re.DOTALL)
        if match:
            try:
                return json.loads(match.group())
            except json.JSONDecodeError:
                pass
    return None

def extract_gt_json(text):
    """Extract the {"triplets": [...]} JSON object from full text using bracket matching."""
    try:
        start = text.rfind('{"triplets":')
        if start < 0:
            return None
        json_start = text.find('[', start)
        if json_start < 0:
            return None
        depth = 1
        pos = json_start + 1
        while pos < len(text) and depth > 0:
            c = text[pos]
            if c == '[':
                depth += 1
            elif c == ']':
                depth -= 1
            pos += 1
        if pos < len(text) and text[pos] == '}':
            return json.loads(text[start:pos+1])
        return None
    except Exception:
        return None

def inference(text, max_new_tokens=512):
    """Run the student model on a single text."""
    prompt = SYSTEM_PROMPT + f"\n\nText: \"{text}\""
    inputs = tokenizer(prompt, return_tensors="pt").to(model_to_eval.device)
    with torch.no_grad():
        output = model_to_eval.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.0,
            do_sample=False,
        )
    generated = tokenizer.decode(output[0], skip_special_tokens=True)
    answer = generated[len(prompt):].strip()
    return extract_json(answer)

# Run on 3 val examples
for i in range(3):
    ex = val_examples[i]
    # Extract original text from flat 'text' field
    m = re.search(r'Text: "([^"]+)"', ex["text"])
    text = m.group(1) if m else "(not found)"
    # Extract ground truth triplets
    gt_parsed = extract_gt_json(ex["text"])
    gt_triplets = gt_parsed['triplets'] if gt_parsed else []

    pred = inference(text)

    print(f"--- Example {i} ---")
    print(f"Ground truth triplets ({len(gt_triplets)}):")
    for t in gt_triplets[:3]:
        print(f"  {t['subject']} | {t['relation']} | {t['object']}")
    print(f"\nPredicted (valid JSON: {pred is not None}):")
    if pred and "triplets" in pred:
        for t in pred["triplets"][:3]:
            print(f"  {t.get('subject','?')} | {t.get('relation','?')} | {t.get('object','?')}")
    else:
        print("  (failed to parse or no triplets)")
    print()
